# Chapter 3 - Tree-Based Benchmark Models (Random Forest and XGBoost)

**Dissertation:** Understanding Health Insurance Cost Predictions Using Deep Neural Networks, Monte Carlo Simulation and Shapley Values  
**Author:** Thabang Bongani Junior Baloyi (2015015486)  
**Supervisor:** Mr J. Blomerous (FASSA)  

This notebook trains Random Forest and XGBoost benchmark models on the same preprocessed data used for the DNN.  
Results are compared against the DNN and GLM in Chapter 4.

## 1. Setup and Data Loading

In [ ]:
import os
import gc
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import xgboost as xgb

warnings.filterwarnings('ignore')

sns.set_style('white')
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'axes.grid': False,
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12
})

FIG_DIR = '/Users/baloyithabangbonganijunior/Downloads/chapter3_figures/'
os.makedirs(FIG_DIR, exist_ok=True)
DATA_DIR = '/Users/baloyithabangbonganijunior/Downloads/'

SEED = 42
np.random.seed(SEED)

print(f'XGBoost version: {xgb.__version__}')
print(f'Seed: {SEED}')

In [ ]:
# Load the same train/val/test partitions used by the DNN
df_train = pd.read_csv(DATA_DIR + 'model_ready_charges_train_20260517_115748.csv')
df_val   = pd.read_csv(DATA_DIR + 'model_ready_charges_vali_20260517_115748.csv')
df_test  = pd.read_csv(DATA_DIR + 'model_ready_charges_test_20260517_115748.csv')

TARGET = 'charges'
FEATURES = [c for c in df_train.columns if c != TARGET]

X_train_raw = df_train[FEATURES].values.astype(np.float32)
y_train_raw = df_train[TARGET].values.astype(np.float32)
X_val_raw = df_val[FEATURES].values.astype(np.float32)
y_val_raw = df_val[TARGET].values.astype(np.float32)
X_test_raw = df_test[FEATURES].values.astype(np.float32)
y_test_raw = df_test[TARGET].values.astype(np.float32)

print(f'Train: {df_train.shape}')
print(f'Val  : {df_val.shape}')
print(f'Test : {df_test.shape}')
print(f'Features ({len(FEATURES)}): {FEATURES}')

# Free DataFrame memory
del df_train, df_val, df_test
gc.collect()

## 2. Input Standardisation

Apply the same standardisation as the DNN notebook (training-set parameters only).

In [ ]:
X_mean = X_train_raw.mean(axis=0)
X_std  = X_train_raw.std(axis=0)
X_std[X_std == 0] = 1.0

X_train = (X_train_raw - X_mean) / X_std
X_val   = (X_val_raw   - X_mean) / X_std
X_test  = (X_test_raw  - X_mean) / X_std

print(f'Standardisation applied (training-set parameters)')
print(f'X_mean (first 5): {X_mean[:5]}')
print(f'X_std  (first 5): {X_std[:5]}')

## 3. XGBoost Benchmark

**Specification:** 1000 boosting rounds, `learning_rate=0.05`, `max_depth=8`, early stopping patience of 50 rounds.  
XGBoost is trained first as it uses less memory than Random Forest.

In [ ]:
print('=' * 70)
print('XGBOOST BENCHMARK')
print('=' * 70)

dtrain = xgb.DMatrix(X_train, label=y_train_raw)
dval = xgb.DMatrix(X_val, label=y_val_raw)

xgb_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'learning_rate': 0.05,
    'max_depth': 8,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'seed': SEED,
    'nthread': 4,
    'verbosity': 0
}

t0 = time.time()
xgb_model = xgb.train(
    xgb_params,
    dtrain,
    num_boost_round=1000,
    evals=[(dtrain, 'train'), (dval, 'val')],
    early_stopping_rounds=50,
    verbose_eval=100
)
xgb_time = time.time() - t0
print(f'\nTraining time: {xgb_time:.1f}s')
print(f'Best iteration: {xgb_model.best_iteration}')

In [ ]:
# XGBoost evaluation
dtest = xgb.DMatrix(X_test, label=y_test_raw)

xgb_pred_train = xgb_model.predict(dtrain)
xgb_pred_val   = xgb_model.predict(dval)
xgb_pred_test  = xgb_model.predict(dtest)

xgb_metrics = {}
print(f'\n{"Set":<12} {"R2":>10} {"RMSE (R)":>12} {"MAE (R)":>12}')
print('-' * 46)
for set_name, y_true, y_pred in [('Train', y_train_raw, xgb_pred_train),
                                   ('Validation', y_val_raw, xgb_pred_val),
                                   ('Test', y_test_raw, xgb_pred_test)]:
    r2 = r2_score(y_true, y_pred)
    rmse_val = np.sqrt(mean_squared_error(y_true, y_pred))
    mae_val = mean_absolute_error(y_true, y_pred)
    xgb_metrics[set_name] = {'R2': r2, 'RMSE': rmse_val, 'MAE': mae_val}
    print(f'{set_name:<12} {r2:>10.6f} {rmse_val:>12.2f} {mae_val:>12.2f}')

# Feature importance (gain)
importance_scores = xgb_model.get_score(importance_type='gain')
xgb_importance = pd.DataFrame({
    'Feature': list(importance_scores.keys()),
    'Importance': list(importance_scores.values())
}).sort_values('Importance', ascending=False)
# Map f0, f1, ... back to feature names
feature_map = {f'f{i}': feat for i, feat in enumerate(FEATURES)}
xgb_importance['Feature'] = xgb_importance['Feature'].map(feature_map)
print('\nXGBoost Feature Importance (gain, top 10):')
print(xgb_importance.head(10).to_string(index=False, float_format='%.2f'))

# Free DMatrix memory
del dtrain, dval
gc.collect()

## 4. Random Forest Benchmark

**Specification:** 500 trees, `max_depth=30`, `min_samples_leaf=5`, `random_state=42`.  
Depth is limited to 30 to control memory usage on large datasets.

In [ ]:
print('=' * 70)
print('RANDOM FOREST BENCHMARK')
print('=' * 70)

t0 = time.time()
rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=30,
    min_samples_leaf=5,
    n_jobs=4,
    random_state=SEED
)
rf.fit(X_train, y_train_raw)
rf_time = time.time() - t0
print(f'Training time: {rf_time:.1f}s')

In [ ]:
# Random Forest evaluation
rf_pred_train = rf.predict(X_train)
rf_pred_val   = rf.predict(X_val)
rf_pred_test  = rf.predict(X_test)

rf_metrics = {}
print(f'\n{"Set":<12} {"R2":>10} {"RMSE (R)":>12} {"MAE (R)":>12}')
print('-' * 46)
for set_name, y_true, y_pred in [('Train', y_train_raw, rf_pred_train),
                                   ('Validation', y_val_raw, rf_pred_val),
                                   ('Test', y_test_raw, rf_pred_test)]:
    r2 = r2_score(y_true, y_pred)
    rmse_val = np.sqrt(mean_squared_error(y_true, y_pred))
    mae_val = mean_absolute_error(y_true, y_pred)
    rf_metrics[set_name] = {'R2': r2, 'RMSE': rmse_val, 'MAE': mae_val}
    print(f'{set_name:<12} {r2:>10.6f} {rmse_val:>12.2f} {mae_val:>12.2f}')

# Feature importance (impurity-based)
rf_importance = pd.DataFrame({
    'Feature': FEATURES,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)
print('\nRF Feature Importance (impurity, top 10):')
print(rf_importance.head(10).to_string(index=False, float_format='%.6f'))

## 5. Four-Model Comparison Summary

Consolidate DNN, GLM, Random Forest and XGBoost test-set metrics.

In [ ]:
# DNN and GLM metrics (from trained models in other notebooks)
# DNN: R2=0.9957, RMSE=291, MAE=251
# GLM: R2=0.9571, RMSE=913, MAE=624

comparison_rows = [
    {'Model': 'GLM (Gamma, log-link)',
     'Test R2': 0.9571, 'Test RMSE': 913, 'Test MAE': 624},
    {'Model': 'Random Forest',
     'Test R2': rf_metrics['Test']['R2'],
     'Test RMSE': rf_metrics['Test']['RMSE'],
     'Test MAE': rf_metrics['Test']['MAE']},
    {'Model': 'XGBoost',
     'Test R2': xgb_metrics['Test']['R2'],
     'Test RMSE': xgb_metrics['Test']['RMSE'],
     'Test MAE': xgb_metrics['Test']['MAE']},
    {'Model': 'DNN (FunnelDNN)',
     'Test R2': 0.9957, 'Test RMSE': 291, 'Test MAE': 251},
]

df_comparison = pd.DataFrame(comparison_rows)
print('=' * 60)
print('FOUR-MODEL COMPARISON (Test Set)')
print('=' * 60)
print(df_comparison.to_string(index=False, float_format='%.4f'))

comparison_path = FIG_DIR + 'four_model_comparison.csv'
df_comparison.to_csv(comparison_path, index=False)
print(f'\nSaved: {comparison_path}')

In [ ]:
# --- Figure: Four-Model Test-Set Performance Comparison ---
# This generates fig_full_model_comparison.png for Chapter 4 (Figure 29)

model_names = ['GLM\n(Gamma, log-link)', 'Random Forest', 'XGBoost', 'DNN\n(FunnelDNN)']

test_r2   = [0.9571,
             rf_metrics['Test']['R2'],
             xgb_metrics['Test']['R2'],
             0.9957]
test_rmse = [913,
             rf_metrics['Test']['RMSE'],
             xgb_metrics['Test']['RMSE'],
             290]
test_mae  = [624,
             rf_metrics['Test']['MAE'],
             xgb_metrics['Test']['MAE'],
             251]

colours = ['#9E9E9E', '#4CAF50', '#FF9800', '#2196F3']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# (a) Test R2
bars = axes[0].bar(model_names, test_r2, color=colours, edgecolor='white', width=0.5)
axes[0].set_ylabel('Test R\u00b2')
axes[0].set_title('(a) Test R\u00b2')
axes[0].set_ylim(0.95, 1.002)
for b, v in zip(bars, test_r2):
    axes[0].text(b.get_x() + b.get_width() / 2, b.get_height() + 0.0005,
                 f'{v:.4f}', ha='center', va='bottom', fontsize=10)
axes[0].tick_params(axis='x', rotation=0, labelsize=9)
sns.despine(ax=axes[0])

# (b) Test RMSE
bars = axes[1].bar(model_names, test_rmse, color=colours, edgecolor='white', width=0.5)
axes[1].set_ylabel('Test RMSE (R)')
axes[1].set_title('(b) Test RMSE')
for b, v in zip(bars, test_rmse):
    axes[1].text(b.get_x() + b.get_width() / 2, b.get_height() + 10,
                 f'R{v:.0f}', ha='center', va='bottom', fontsize=10)
axes[1].tick_params(axis='x', rotation=0, labelsize=9)
sns.despine(ax=axes[1])

# (c) Test MAE
bars = axes[2].bar(model_names, test_mae, color=colours, edgecolor='white', width=0.5)
axes[2].set_ylabel('Test MAE (R)')
axes[2].set_title('(c) Test MAE')
for b, v in zip(bars, test_mae):
    axes[2].text(b.get_x() + b.get_width() / 2, b.get_height() + 5,
                 f'R{v:.0f}', ha='center', va='bottom', fontsize=10)
axes[2].tick_params(axis='x', rotation=0, labelsize=9)
sns.despine(ax=axes[2])

plt.suptitle('Model Comparison: Test Set Performance', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR + 'fig_full_model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR}fig_full_model_comparison.png')

## 6. Diagnostic Figures

In [ ]:
# --- Figure: Feature Importance Comparison ---
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

rf_imp_sorted = rf_importance.sort_values('Importance', ascending=True)
axes[0].barh(rf_imp_sorted['Feature'], rf_imp_sorted['Importance'],
             color='#4CAF50', edgecolor='white')
axes[0].set_xlabel('Impurity-Based Importance')
axes[0].set_title('Random Forest Feature Importance')
sns.despine(ax=axes[0])

xgb_imp_sorted = xgb_importance.sort_values('Importance', ascending=True)
axes[1].barh(xgb_imp_sorted['Feature'], xgb_imp_sorted['Importance'],
             color='#FF9800', edgecolor='white')
axes[1].set_xlabel('Gain-Based Importance')
axes[1].set_title('XGBoost Feature Importance')
sns.despine(ax=axes[1])

plt.suptitle('Tree-Based Model Feature Importance', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR + 'fig_tree_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR}fig_tree_feature_importance.png')

In [ ]:
# --- Figure: Actual vs Predicted Scatter (Test Set, all models) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

models_preds = [
    ('Random Forest', rf_pred_test, '#4CAF50'),
    ('XGBoost', xgb_pred_test, '#FF9800')
]

for i, (name, preds, colour) in enumerate(models_preds):
    ax = axes[i]
    n_plot = min(5000, len(y_test_raw))
    idx = np.random.choice(len(y_test_raw), n_plot, replace=False)

    r2 = r2_score(y_test_raw[idx], preds[idx])
    ax.scatter(y_test_raw[idx], preds[idx], alpha=0.3, s=5, color=colour)

    lims = [min(y_test_raw.min(), preds.min()), max(y_test_raw.max(), preds.max())]
    ax.plot(lims, lims, 'r--', linewidth=1, alpha=0.7)

    ax.set_xlabel('Actual Charges (R)')
    ax.set_ylabel('Predicted Charges (R)')
    ax.set_title(f'{name} (R\u00b2 = {r2:.4f})')
    sns.despine(ax=ax)

plt.suptitle('Actual vs Predicted Charges (Test Set)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR + 'fig_tree_model_scatter.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR}fig_tree_model_scatter.png')

In [ ]:
# --- Save feature importance CSVs ---
rf_importance.to_csv(FIG_DIR + 'rf_feature_importance.csv', index=False)
xgb_importance.to_csv(FIG_DIR + 'xgb_feature_importance.csv', index=False)
print(f'RF importance saved: {FIG_DIR}rf_feature_importance.csv')
print(f'XGBoost importance saved: {FIG_DIR}xgb_feature_importance.csv')

# --- LaTeX table values ---
print('\n' + '=' * 60)
print('LATEX TABLE VALUES')
print('=' * 60)

print('\nRandom Forest')
for name in ['Train', 'Validation', 'Test']:
    m = rf_metrics[name]
    label = {'Train': 'Training', 'Validation': 'Validation', 'Test': 'Test'}[name]
    print(f'    & {label}   & {m["MAE"]:.0f} & {m["RMSE"]:.0f} & {m["R2"]:.4f} \\\\')

print('\nXGBoost')
for name in ['Train', 'Validation', 'Test']:
    m = xgb_metrics[name]
    label = {'Train': 'Training', 'Validation': 'Validation', 'Test': 'Test'}[name]
    print(f'    & {label}   & {m["MAE"]:.0f} & {m["RMSE"]:.0f} & {m["R2"]:.4f} \\\\')

In [ ]:
# --- Final Summary ---
print('=' * 70)
print('TREE-BASED BENCHMARK MODELS COMPLETE')
print('=' * 70)
print()
print('Random Forest:')
print(f'  Trees        : 500')
print(f'  Max depth    : 30')
print(f'  Test R2      : {rf_metrics["Test"]["R2"]:.4f}')
print(f'  Test RMSE    : R{rf_metrics["Test"]["RMSE"]:,.2f}')
print(f'  Test MAE     : R{rf_metrics["Test"]["MAE"]:,.2f}')
print()
print('XGBoost:')
print(f'  Boosting rds : {xgb_model.best_iteration}')
print(f'  Learning rate: 0.05')
print(f'  Max depth    : 8')
print(f'  Test R2      : {xgb_metrics["Test"]["R2"]:.4f}')
print(f'  Test RMSE    : R{xgb_metrics["Test"]["RMSE"]:,.2f}')
print(f'  Test MAE     : R{xgb_metrics["Test"]["MAE"]:,.2f}')
print()
print('Figures generated:')
for f in ['fig_full_model_comparison.png', 'fig_tree_feature_importance.png',
          'fig_tree_model_scatter.png']:
    print(f'  - {f}')
print()
print('CSV exports:')
for f in ['four_model_comparison.csv', 'rf_feature_importance.csv', 'xgb_feature_importance.csv']:
    print(f'  - {f}')